# 📘 Univariate Linear Regression Using Gradient Descent
### Dataset: Student Performance — UCI ML Repository (id=320)
> Built from scratch using **NumPy**, **Pandas**, and **Matplotlib** only.

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ucimlrepo import fetch_ucirepo

%matplotlib inline

## 2. Load Dataset from UCI ML Repository

In [ ]:
# Fetch Student Performance dataset (id=320)
student_performance = fetch_ucirepo(id=320)

X_data = student_performance.data.features
y_data = student_performance.data.targets

# Merge into one dataframe for EDA
df = pd.concat([X_data, y_data], axis=1)
print("Dataset loaded successfully!")

## 3. Dataset Metadata & Variable Info

In [ ]:
print("Metadata:\n")
print(student_performance.metadata)

print("\nVariable Information:\n")
print(student_performance.variables)

## 4. Basic Exploratory Data Analysis (EDA)

In [ ]:
df.head()

In [ ]:
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())

In [ ]:
print("Missing Values:\n")
print(df.isnull().sum())

In [ ]:
df.describe()

## 5. Select Numeric Columns

In [ ]:
numeric_columns = df.select_dtypes(include=np.number)
numeric_columns.head()

## 6. Correlation Analysis

**G3** is the final student grade — our prediction target.

| Value | Meaning |
|-------|---------|
| `+1`  | Strong positive relationship |
| `-1`  | Strong negative relationship |
| `0`   | No relationship |

In [ ]:
correlation = numeric_columns.corr()['G3']
print("Correlation with G3:\n")
print(correlation)

In [ ]:
correlation = correlation.drop('G3')
correlation = correlation.sort_values(key=abs, ascending=False)

FEATURE = correlation.index[0]
print("Selected Feature:", FEATURE)

## 7. Visualise Correlation (Scatter Plot)

In [ ]:
plt.figure(figsize=(6, 4))
plt.scatter(df[FEATURE], df['G3'], alpha=0.6, color='steelblue',
            edgecolors='k', linewidths=0.4)
plt.xlabel(FEATURE, fontsize=12)
plt.ylabel("G3", fontsize=12)
plt.title(f"{FEATURE} vs G3", fontsize=14)
plt.tight_layout()
plt.savefig("plots/scatter_correlation.png", dpi=150)
plt.show()

## 8. Prepare Data

In [ ]:
X = df[FEATURE].values.astype(float)
y = df['G3'].values.astype(float)
n = len(X)
print(f"Samples: {n}  |  Feature: {FEATURE}")

## 9. Feature Scaling (Standardisation)

$$X_{\text{scaled}} = \frac{X - \mu}{\sigma}$$

Scaling ensures gradient descent converges faster and more reliably.

In [ ]:
X_mean = np.mean(X)
X_std  = np.std(X)
X = (X - X_mean) / X_std

print(f"Post-scaling → Mean: {np.mean(X):.4f}  |  Std: {np.std(X):.4f}")

## 10. Initialise Parameters

Model: $\hat{y} = wX + b$

Starting with `w = 0`, `b = 0`.

In [ ]:
w = 0.0
b = 0.0
learning_rate = 0.01
iterations    = 1000
loss_history  = []

print(f"w = {w}, b = {b}, lr = {learning_rate}, iters = {iterations}")

## 11. Gradient Descent

$$\frac{\partial L}{\partial w} = \frac{2}{n}\sum(\hat{y}-y)\cdot X \qquad
\frac{\partial L}{\partial b} = \frac{2}{n}\sum(\hat{y}-y)$$

In [ ]:
for i in range(iterations):
    predictions = w * X + b
    error       = predictions - y

    dw = (2/n) * np.sum(error * X)
    db = (2/n) * np.sum(error)

    w = w - learning_rate * dw
    b = b - learning_rate * db

    loss_history.append(np.mean(error ** 2))

print("Training complete!")
print(f"Final w = {w:.4f}  |  Final b = {b:.4f}")

## 12. Make Predictions

In [ ]:
predictions = w * X + b

## 13. Evaluate — R² Score

$$R^2 = 1 - \frac{\sum(y - \hat{y})^2}{\sum(y - \bar{y})^2}$$

In [ ]:
residual_error = np.sum((y - predictions) ** 2)
total_error    = np.sum((y - np.mean(y)) ** 2)
r2_score       = 1 - (residual_error / total_error)

print(f"R² Score: {r2_score:.4f}")

## 14. Loss Curve

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(loss_history, color='crimson', linewidth=1.5)
plt.xlabel("Iterations", fontsize=12)
plt.ylabel("MSE Loss", fontsize=12)
plt.title("Loss vs Iterations", fontsize=14)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("plots/loss_curve.png", dpi=150)
plt.show()

## 15. Regression Line Plot

In [ ]:
sort_idx = np.argsort(X)

plt.figure(figsize=(8, 5))
plt.scatter(X, y, alpha=0.6, color='steelblue',
            edgecolors='k', linewidths=0.4, label="Actual Data")
plt.plot(X[sort_idx], predictions[sort_idx],
         color='red', linewidth=2, label="Regression Line")
plt.xlabel(f"{FEATURE} (scaled)", fontsize=12)
plt.ylabel("G3", fontsize=12)
plt.title("Linear Regression Fit", fontsize=14)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("plots/regression_line.png", dpi=150)
plt.show()

## 16. Residual Plot

Residuals = Actual − Predicted.  
A good model shows residuals scattered **randomly around zero** with no pattern.

In [ ]:
residuals = y - predictions

plt.figure(figsize=(8, 5))
plt.scatter(predictions, residuals, alpha=0.6, color='darkorange',
            edgecolors='k', linewidths=0.4)
plt.axhline(y=0, color='black', linewidth=1.2, linestyle='--')
plt.xlabel("Predicted Values", fontsize=12)
plt.ylabel("Residuals", fontsize=12)
plt.title("Residual Plot", fontsize=14)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("plots/residual_plot.png", dpi=150)
plt.show()

## 17. Conclusion

| Item | Detail |
|------|--------|
| **Dataset** | UCI Student Performance (id=320) |
| **Target** | G3 — Final student grade |
| **Feature** | Auto-selected via highest absolute correlation |
| **Algorithm** | Manual Gradient Descent (no sklearn) |
| **Metric** | R² Score |

> Gradient Descent iteratively minimises MSE by nudging `w` and `b` in the direction that reduces error.  
> Feature standardisation is essential for stable and fast convergence.